# Task 2: Preprocessing, Model Training and Evaluation
**CM2604 Machine Learning: Coursework Part 01**

Prepares the data and compares **KNN**, **Logistic Regression** and **Random Forest** in two settings: baseline (default hyperparameters) and tuned (GridSearchCV).

## Setup and data loading

In [ ]:
# Import all libraries used in this notebook
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay,
                             accuracy_score, classification_report,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "Churn"

# Folders for saving figures and results (downloaded at the end)
FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def save(fig, name):
    """Save a figure to figures/ and show it in the notebook."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

The dataset is loaded from the `data/` folder of the project's public GitHub repository.
**Change `GITHUB_USERNAME` below to your own GitHub username.**

In [ ]:
GITHUB_USERNAME = "yasiru20250101"         # your GitHub username
REPO_NAME = "telco-churn-classification"
DATA_URL = (f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}"
            "/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

raw = pd.read_csv(DATA_URL)
print("Shape:", raw.shape)
raw.head()

## Data cleaning

In [ ]:
print("Data types:")
print(raw.dtypes)
print("\nMissing values (NaN):", raw.isna().sum().sum())

# TotalCharges is stored as text: check for blank strings
blank_tc = raw["TotalCharges"].astype(str).str.strip() == ""
print("Blank strings in TotalCharges:", blank_tc.sum())
print("Tenure of those customers:", raw.loc[blank_tc, "tenure"].unique())

In [ ]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]


def clean_data(df):
    """
    1. Drop customerID (unique identifier, no predictive value).
    2. Convert TotalCharges to numeric; blanks (tenure = 0) become NaN.
    3. Fill those NaN with 0 because new customers have not been charged yet.
    4. Convert SeniorCitizen 0/1 to No/Yes so it is treated as categorical.
    5. Encode the target: Yes -> 1, No -> 0.
    6. Remove duplicate rows.
    """
    df = df.copy()
    df = df.drop(columns=["customerID"])
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(),
                                       errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(0)
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
    df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicate rows removed: {before - len(df)}")
    return df


df = clean_data(raw)
print("Clean shape:", df.shape)
df[NUMERIC_FEATURES].describe().round(2)

## Preprocessing pipeline

In [ ]:
# Stratified 80/20 split keeps the same churn ratio in train and test sets
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))


def build_preprocessor(scale=True):
    """
    Numeric     -> StandardScaler (KNN and LR need scaling; RF does not -> scale=False)
    Categorical -> One-Hot Encoding (drop='if_binary' keeps one column for Yes/No)
    Used inside a Pipeline so it is fitted on the training data only (no leakage).
    """
    numeric_step = StandardScaler() if scale else "passthrough"
    return ColumnTransformer(transformers=[
        ("num", numeric_step, NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"),
         CATEGORICAL_FEATURES),
    ])


n_features = len(build_preprocessor().fit(X_train).get_feature_names_out())
print("Number of features after encoding:", n_features)

## Baseline and tuned models

Each model is run in two experimental settings:
1. **Baseline**: default hyperparameters
2. **Tuned**: best hyperparameters from `GridSearchCV` (5-fold stratified CV, optimising F1)

In [ ]:
SCORING = "f1"
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "KNN": (
        Pipeline([("pre", build_preprocessor(scale=True)),
                  ("clf", KNeighborsClassifier())]),
        {
            "clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"],
        },
    ),
    "Logistic Regression": (
        Pipeline([("pre", build_preprocessor(scale=True)),
                  ("clf", LogisticRegression(max_iter=2000,
                                             random_state=RANDOM_STATE))]),
        {
            # C = inverse of regularisation strength (L2 regularisation)
            "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        Pipeline([("pre", build_preprocessor(scale=False)),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE,
                                                 n_jobs=-1))]),
        {
            "clf__n_estimators": [100, 300],
            "clf__max_depth": [None, 6, 10, 15],
            "clf__min_samples_split": [2, 10],
            "clf__min_samples_leaf": [1, 4],
            "clf__class_weight": [None, "balanced"],
        },
    ),
}


def evaluate(name, setting, model):
    """Return test-set metrics for one fitted model."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n===== {name} ({setting}) =====")
    print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))
    return {
        "Model": name,
        "Setting": setting,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
    }

In [ ]:
# Training takes a few minutes (Random Forest grid is the largest)
results = []
tuned_models = {}
best_params = {}

for name, (pipe, grid) in models.items():
    # Setting 1: baseline
    pipe.fit(X_train, y_train)
    results.append(evaluate(name, "Baseline", pipe))

    # Setting 2: hyperparameter tuning
    search = GridSearchCV(pipe, grid, scoring=SCORING, cv=CV, n_jobs=-1)
    search.fit(X_train, y_train)
    print(f"[{name}] Best params: {search.best_params_}")
    print(f"[{name}] Best CV {SCORING}: {search.best_score_:.4f}")
    best_params[name] = search.best_params_
    tuned_models[name] = search.best_estimator_
    results.append(evaluate(name, "Tuned", search.best_estimator_))

## Evaluation and comparison

In [ ]:
results_df = pd.DataFrame(results).round(4)
results_df.to_csv(RESULTS_DIR / "model_results.csv", index=False)

with open(RESULTS_DIR / "best_params.txt", "w") as f:
    for name, params in best_params.items():
        f.write(f"{name}: {params}\n")

print("Best hyperparameters:")
for name, params in best_params.items():
    print(f"  {name}: {params}")
results_df

In [ ]:
# Baseline vs tuned comparison
melted = results_df.melt(id_vars=["Model", "Setting"],
                         value_vars=["F1", "Recall", "ROC_AUC"],
                         var_name="Metric", value_name="Score")
g = sns.catplot(data=melted, x="Model", y="Score", hue="Setting",
                col="Metric", kind="bar", height=4, aspect=1)
g.set_xticklabels(rotation=20)
g.figure.suptitle("Baseline vs Tuned models (test set)", y=1.04)
g.savefig(FIG_DIR / "08_baseline_vs_tuned.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ROC curves of the tuned models
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in tuned_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title("ROC curves (tuned models)")
ax.legend()
save(fig, "09_roc_curves.png")

# Confusion matrices of the tuned models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, tuned_models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax,
                                          display_labels=["No", "Yes"],
                                          cmap="Blues", colorbar=False)
    ax.set_title(name)
save(fig, "10_confusion_matrices.png")

## Explainability: which features drive the predictions?

In [ ]:
# Random Forest feature importance
rf = tuned_models["Random Forest"]
feat_names = rf.named_steps["pre"].get_feature_names_out()
imp = (pd.Series(rf.named_steps["clf"].feature_importances_, index=feat_names)
       .sort_values(ascending=False).head(15))
fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(x=imp.values, y=imp.index, ax=ax, color="seagreen")
ax.set_title("Top 15 features - Random Forest")
save(fig, "11_rf_feature_importance.png")

# Logistic Regression coefficients (explainability)
lr = tuned_models["Logistic Regression"]
coefs = pd.Series(lr.named_steps["clf"].coef_[0],
                  index=lr.named_steps["pre"].get_feature_names_out())
top = pd.concat([coefs.nlargest(8), coefs.nsmallest(8)]).sort_values()
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top.index, top.values,
        color=["indianred" if v > 0 else "steelblue" for v in top.values])
ax.set_title("Logistic Regression coefficients\n(red = increases churn risk)")
save(fig, "12_lr_coefficients.png")

## Best model

In [ ]:
# Select the best tuned model by F1-score
tuned = results_df[results_df["Setting"] == "Tuned"]
best_name = tuned.loc[tuned["F1"].idxmax(), "Model"]
best_model = tuned_models[best_name]
print("Best model by F1:", best_name)

## Download figures and results

In [ ]:
# Zip the figures/ and results/ folders and download them for the report
import shutil
shutil.make_archive("task2_figures", "zip", ".", "figures")
shutil.make_archive("task2_results", "zip", ".", "results")

try:
    from google.colab import files
    files.download("task2_figures.zip")
    files.download("task2_results.zip")
except ImportError:
    print("Not running in Colab: zip files saved in the current folder.")